In [ ]:
# Ejecuta esta celda para instalar todas las librerías necesarias
!pip install tensorflow pillow matplotlib scipy scikit-learn pandas

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import preprocess_input as preprocess_vgg16
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_mobilenetv2
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize


In [ ]:
# === Parameters ===
IMG_WIDTH, IMG_HEIGHT = 224, 224
BATCH_SIZE = 32
BASE_DIR = Path.cwd()
TEST_DATA_DIR = BASE_DIR / "Animals_Dataset_Splitted" / "test"

# === Data Generators ===
# Custom model has Rescaling(1./255) built-in, so no preprocessing function needed
test_datagen_custom = ImageDataGenerator()
test_generator_custom = test_datagen_custom.flow_from_directory(
    TEST_DATA_DIR,
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode="sparse",
    shuffle=False
)

# VGG16 preprocessing
test_datagen_vgg16 = ImageDataGenerator(preprocessing_function=preprocess_vgg16)
test_generator_vgg16 = test_datagen_vgg16.flow_from_directory(
    TEST_DATA_DIR,
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode="sparse",
    shuffle=False
)

# MobileNetV2 preprocessing
test_datagen_mobilenet = ImageDataGenerator(preprocessing_function=preprocess_mobilenetv2)
test_generator_mobilenet = test_datagen_mobilenet.flow_from_directory(
    TEST_DATA_DIR,
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode="sparse",
    shuffle=False
)

class_names = list(test_generator_custom.class_indices.keys())


In [ ]:
# === Load Models ===
print("Cargando modelo Custom...")
model_custom = load_model("modelo_peces_Custom.keras")

print("Cargando modelo VGG16...")
model_vgg16 = load_model("modelo_peces_vgg16.keras")

print("Cargando modelo MobileNetV2...")
model_mobilenet = load_model("modelo_peces_MobileNetV2.keras")



In [ ]:
# === Evaluate Models ===
print("\nEvaluando modelo Custom...")
loss_custom, acc_custom = model_custom.evaluate(test_generator_custom, verbose=1)

print("\nEvaluando modelo VGG16...")
loss_vgg16, acc_vgg16 = model_vgg16.evaluate(test_generator_vgg16, verbose=1)

print("\nEvaluando modelo MobileNetV2...")
loss_mobilenet, acc_mobilenet = model_mobilenet.evaluate(test_generator_mobilenet, verbose=1)

print("\n=== Resultados de Evaluación en Test ===")
print(f"Custom      - Accuracy: {acc_custom:.4f}, Loss: {loss_custom:.4f}")
print(f"VGG16       - Accuracy: {acc_vgg16:.4f}, Loss: {loss_vgg16:.4f}")
print(f"MobileNetV2       - Accuracy: {acc_mobilenet:.4f}, Loss: {loss_mobilenet:.4f}")

In [ ]:
# === Graph Accuracy Comparison ===
models = ['Custom', 'VGG16', 'MobileNetV2']
accuracies = [acc_custom, acc_vgg16, acc_mobilenet]

plt.figure(figsize=(8, 5))
bars = plt.bar(models, accuracies, color=['skyblue', 'lightgreen', 'salmon'])
plt.ylim(0, 1.1)
plt.title('Comparación de Accuracy en Datos de Test')
plt.ylabel('Accuracy')

# Añadir etiquetas de datos
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f"{yval:.4f}", ha='center', va='bottom', fontsize=12, fontweight='bold')



In [ ]:
# === Evaluación Avanzada y Matrices de Confusión ===
# Priorizamos F1-Score Macro debido a que es una clasificación multiclase donde cada imagen 
# corresponde a una única especie de pez, permitiendo una métrica robusta ante posibles desbalances.

def evaluate_model_advanced(model, generator, title, class_names):
    print(f"\n{'='*50}")
    print(f"Evaluando {title}...")
    print(f"{'='*50}")
    
    generator.reset()
    preds = model.predict(generator, verbose=1)
    y_pred = np.argmax(preds, axis=1)
    y_true = generator.classes
    
    # Matriz de confusión
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(8, 8))
    disp.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation=45)
    plt.title(f'Matriz de Confusión: {title}')
    plt.show()
    
    # Classification Report
    print(f"\nClassification Report: {title}\n")
    print(classification_report(y_true, y_pred, target_names=class_names))
    
    metrics = {'Model': title}
    
    # Accuracy
    try:
        metrics['Accuracy'] = accuracy_score(y_true, y_pred)
    except Exception:
        metrics['Accuracy'] = np.nan
        
    # Precision Macro
    try:
        metrics['Precision Macro'] = precision_score(y_true, y_pred, average='macro')
    except Exception:
        metrics['Precision Macro'] = np.nan

    # Recall Macro
    try:
        metrics['Recall Macro'] = recall_score(y_true, y_pred, average='macro')
    except Exception:
        metrics['Recall Macro'] = np.nan

    # F1 Macro
    try:
        metrics['F1 Macro'] = f1_score(y_true, y_pred, average='macro')
    except Exception:
        metrics['F1 Macro'] = np.nan

    # Specificity Macro
    try:
        # Specificity = TN / (TN + FP)
        # Per class specificity
        specificity_list = []
        for i in range(len(class_names)):
            tn = np.sum(np.delete(np.delete(cm, i, 0), i, 1))
            fp = np.sum(cm[:, i]) - cm[i, i]
            specificity_list.append(tn / (tn + fp) if (tn + fp) > 0 else 0)
        metrics['Specificity'] = np.mean(specificity_list)
    except Exception:
        metrics['Specificity'] = np.nan

    # ROC-AUC
    try:
        y_true_bin = label_binarize(y_true, classes=range(len(class_names)))
        n_classes = y_true_bin.shape[1]
        
        # Calculate ROC-AUC One-vs-Rest
        metrics['ROC-AUC'] = roc_auc_score(y_true_bin, preds, multi_class='ovr')
        
        # Plot ROC curves
        plt.figure(figsize=(10, 8))
        for i in range(n_classes):
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], preds[:, i])
            plt.plot(fpr, tpr, label=f'Clase {class_names[i]}')
        plt.plot([0, 1], [0, 1], 'k--', lw=2)
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('Tasa de Falsos Positivos')
        plt.ylabel('Tasa de Verdaderos Positivos')
        plt.title(f'Curvas ROC One-vs-Rest: {title}')
        plt.legend(loc="lower right")
        plt.show()
    except Exception:
        metrics['ROC-AUC'] = np.nan
        print(f"No se pudo calcular ROC-AUC para {title}.")

    return metrics

results = []
results.append(evaluate_model_advanced(model_custom, test_generator_custom, "Custom Model", class_names))
results.append(evaluate_model_advanced(model_vgg16, test_generator_vgg16, "VGG16", class_names))
results.append(evaluate_model_advanced(model_mobilenet, test_generator_mobilenet, "MobileNetV2", class_names))



In [ ]:
# === Comparación Consolidada ===

df_results = pd.DataFrame(results)

# Renombrar columnas según lo solicitado
df_results = df_results.rename(columns={'Model': 'Modelo'})

print("\nTabla Comparativa de Modelos:")
display(df_results)